# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "I love learning about neural embeddings!"
print(sample_sentence)


I love learning about neural embeddings!


In [4]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | i            |  1045
    2 | love         |  2293
    3 | learning     |  4083
    4 | about        |  2055
    5 | neural       | 15756
    6 | em           |  7861
    7 | ##bed        |  8270
    8 | ##ding       |  4667
    9 | ##s          |  2015
   10 | !            |   999
   11 | [SEP]        |   102
   12 | [PAD]        |     0
   13 | [PAD]        |     0
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (11, '[SEP]'), (12, '[PAD]'), (13, '[PAD]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]

### Exercise 1 reflection
- TODO: Describe how [CLS] and [SEP] behave inside the encoder.
- TODO: Explain how the attention mask hides padded positions from self-attention.


To understand how BERT processes your tokenized input, we need to look at how these markers function within the Transformer's self-attention mechanism.

---

## 1. How [CLS] and [SEP] Behave Inside the Encoder

In the BERT architecture, these tokens aren't just labels; they serve specific structural purposes during the multi-head attention process.

### The [CLS] Token (The "Aggregator")
The `[CLS]` token is placed at index 0. Because BERT uses **self-attention**, every token "looks" at every other token.
*   **Contextual Pooling:** As the input passes through the encoder layers (usually 12 or 24), the `[CLS]` token collects information from every other token in the sequence.
*   **The Final Output:** By the time it reaches the final layer, the vector representation of `[CLS]` is considered a summary of the entire sequence. This is why we use only this specific vector for classification tasks (like sentiment analysis or spam detection).

### The [SEP] Token (The "Boundary")
The `[SEP]` token acts as a hard signal to the model that a segment has ended.
*   **Segment Distinction:** In tasks involving sentence pairs (like Question Answering), `[SEP]` tells the model where the "Context" ends and the "Question" begins.
*   **Relational Encoding:** It helps the self-attention mechanism learn relationships *between* segments. The model learns that tokens on one side of a `[SEP]` may have a different structural relationship to tokens on the other side.



---

## 2. How the Attention Mask Hides Padded Positions

Deep learning models require "rectangular" data (batches of sequences that are all the exact same length). However, sentences vary in size. We use `[PAD]` tokens to fill the gaps, but we don't want the model to actually "learn" from those zeros.

### The Mechanism of the Mask
The **Attention Mask** is a binary tensor (1s for real tokens, 0s for padding) that acts as a filter during the calculation of **Self-Attention**.

1.  **The Score Calculation:** Inside the encoder, the model calculates attention scores between all tokens using a dot-product (Query $\times$ Key).
2.  **The "Masking" Step:** Before applying the Softmax function (which turns scores into probabilities), the model looks at the attention mask.
3.  **Large Negative Values:** For any position where the mask is `0`, the model replaces the attention score with a very large negative number (e.g., $-10^9$).
4.  **The Softmax Effect:** When the Softmax function is applied to these large negative numbers, they effectively become $0$.



> **The Result:** The model effectively "blinds" itself to the padded areas. When a real word (like "Learning") looks around for context, it sees the other words in the sentence but perceives the `[PAD]` tokens as having zero importance. This prevents the padding from diluting the gradients during training.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [5]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "The movie was surprisingly long, but the acting was absolutely world-class."
prediction = sentiment_pipeline(sentence)
prediction


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9995583891868591}]

### Exercise 2 reflection
- TODO: Does the predicted label match your expectation? Why or why not?
- TODO: How confident is the model and what does the score tell you?

Based on the sentence from the previous step (**"The movie was surprisingly long, but the acting was absolutely world-class."**), here is how to interpret the model's performance:

### 1. Does the predicted label match your expectation?
**Yes, it typically does.**
While the sentence contains a "negative" element (the movie being "surprisingly long"), the phrase "absolutely world-class" carries much stronger emotional weight.

*   **Why?** BERT’s self-attention mechanism allows it to understand that "world-class" is the dominant descriptor for the subject. In the encoder, the **[CLS]** token aggregates the "long" and "world-class" embeddings. Because the model was fine-tuned on the SST-2 dataset (which consists of movie reviews), it has learned that high-praise adjectives like "world-class" are stronger indicators of a positive review than length-related complaints.

### 2. How confident is the model and what does the score tell you?
If you ran the code, you likely saw a `score` around **0.999**.

*   **What the score means:** This is the result of the **Softmax** function applied to the model's raw output (logits). A score of 0.99 means the model is 99% confident that the text belongs to the "POSITIVE" class.
*   **The Probability Gap:** If a sentence is ambiguous (e.g., "The movie was okay, I guess."), you might see a score closer to **0.51**. This tells you the model is "confused" or that the sentiment is neutral, as it’s almost equally split between the two categories.
*   **A Note on "Truth":** It is important to remember that this score represents **mathematical confidence**, not objective truth. It simply means "Based on the patterns I saw in my training data, this looks 99% like the other positive examples I've seen."



---

### Key Takeaway for your Analysis
When working with these models, always check the score. If the model gives a label you didn't expect but the score is low (e.g., 0.52), it’s a sign that the language is too nuanced or "gray" for the model's current training. If the score is high (0.99) but the label is wrong, it indicates a **misalignment**—likely due to sarcasm or complex linguistic structures that the model isn't yet equipped to handle.


## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        '''TODO: load the tokenizer/model and move the model to the proper device.'''
        # 1. Load the tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.max_length = max_length

        # 2. Move model to GPU if available, otherwise CPU
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval() # Set to evaluation mode (disables dropout, etc.)

        #raise NotImplementedError("Initialize tokenizer, model, and device here.")

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        '''TODO: clean the text, tokenize, and return tensors ready for inference.'''
        # Clean the text and tokenize with padding/truncation
        # return_tensors='pt' gives us PyTorch tensors immediately
        encoded_input = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        # Move tensors to the same device as the model
        return {k: v.to(self.device) for k, v in encoded_input.items()}
        #raise NotImplementedError("Return a dict of tensors produced by the tokenizer.")

    def predict(self, text: str) -> Dict[str, float]:
        '''TODO: run a forward pass, apply softmax, and return a label plus probability.'''
        # Ensure no gradients are calculated during inference (saves memory/time)
        with torch.no_grad():
            # 1. Preprocess the input
            inputs = self.preprocess(text)

            # 2. Forward pass (returns raw logits)
            outputs = self.model(**inputs)
            logits = outputs.logits

            # 3. Apply Softmax to get probabilities (dim=1 corresponds to the labels)
            probabilities = torch.softmax(logits, dim=1).squeeze()

            # 4. Extract label and confidence
            label_id = torch.argmax(probabilities).item()
            label = self.model.config.id2label[label_id]
            confidence = probabilities[label_id].item()

        return {"label": label, "probability": round(confidence, 4)}
        #raise NotImplementedError("Add inference and post-processing logic.")


In [13]:
# TODO: instantiate your analyzer and test several sentences once the class is ready.
analyzer = BERTSentimentAnalyzer()

samples = [
    "This tutorial is exceptionally clear and helpful for my project!",
    "I'm quite frustrated because the documentation is confusing.",
    "The weather is currently 25 degrees." # A neutral-ish baseline
]

for text in samples:
    result = analyzer.predict(text)
    print(f"Text: {text}")
    print(f"Result: {result}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: This tutorial is exceptionally clear and helpful for my project!
Result: {'label': 'POSITIVE', 'probability': 0.9998}

Text: I'm quite frustrated because the documentation is confusing.
Result: {'label': 'NEGATIVE', 'probability': 0.9997}

Text: The weather is currently 25 degrees.
Result: {'label': 'POSITIVE', 'probability': 0.9843}



## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [14]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        # 1. Load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)

        # 2. Setup device
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

    def recognize(self, text: str):
        # 3. Tokenize input
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)

        # 4. Get the most likely label ID for each token
        predictions = torch.argmax(outputs.logits, dim=2).squeeze().tolist()
        tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())

        # Mapping IDs to human labels (B-PER, I-ORG, etc.)
        id2label = self.model.config.id2label

        entities = []
        current_entity = None

        for token, pred_id in zip(tokens, predictions):
            label = id2label[pred_id]

            # Skip special tokens
            if token in ["[CLS]", "[SEP]", "[PAD]"]:
                continue

            # Check if this is a subword (starts with ##)
            is_subword = token.startswith("##")
            clean_token = token.replace("##", "")

            # Logic to merge subwords or start new entities
            if label != "O":  # "O" means outside an entity
                if is_subword and current_entity:
                    current_entity["text"] += clean_token
                else:
                    # If it's a new word or a new entity type, save the old one
                    if current_entity:
                        entities.append(current_entity)

                    current_entity = {
                        "text": clean_token,
                        "entity": label.split("-")[-1], # Remove B- or I- prefix
                    }
            else:
                if current_entity:
                    entities.append(current_entity)
                    current_entity = None

        # Catch the last entity if one exists
        if current_entity:
            entities.append(current_entity)

        return entities


In [15]:
# TODO: instantiate the recognizer and test it on text that includes people, places, or organizations.
ner = BERTNamedEntityRecognizer()
sample_text = "The Hebrew University is located in Jerusalem and specializes in microelectronics."
results = ner.recognize(sample_text)

for res in results:
    print(res)


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'text': 'Hebrew', 'entity': 'ORG'}
{'text': 'University', 'entity': 'ORG'}
{'text': 'Jerusalem', 'entity': 'LOC'}


## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Bi-directional; sees the entire context simultaneously | Uni-directional; sees only past tokens (causal). |
| Primary purpose | Understanding and extracting meaning from text. | Generating coherent and creative new text. |
| Typical use cases | "Classification, NER, and Sentiment Analysis." | "Chatbots, Creative Writing, and Code Completion." |
| Strengths | Superior at understanding nuanced context and intent. | Excellent at maintaining long-range flow and fluency. |
| Weaknesses | Cannot generate text effectively (sentence-by-sentence). | "Can ""hallucinate"" or drift from factual accuracy." |


## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. TODO: Describe how BERT encodes queries and documents.
2. TODO: Explain how those embeddings are stored and searched in a vector database.
3. TODO: Outline how the retrieved passages are handed to a generative model like GPT.
4. TODO: Provide a concrete application example (industry or product) where RAG with BERT makes sense.


Retrieval-Augmented Generation (RAG) is the bridge between static language models and live, private data. While generative models like GPT handle the "speaking," BERT-style models are the "librarians" that find the right information.

### BERT for Encoding Queries and Documents
In a RAG workflow, BERT (or a similar bi-directional encoder) acts as the embedding engine. It processes raw text—both your knowledge base documents and the user's incoming query—and transforms them into high-dimensional vectors (numerical representations). Because BERT is bi-directional, these embeddings capture the **semantic intent** rather than just keywords. For instance, it understands that a document about "adjusting voltage" is relevant to a query about "electrical regulation" even if the specific words don't match.



### Vector Storage and Similarity Search
Once the documents are converted into vectors, they are stored in a **Vector Database** (like Pinecone, Milvus, or FAISS). When a user asks a question, BERT encodes that question into a vector in the same mathematical space. The database then performs a "Similarity Search"—usually using **Cosine Similarity**—to find the document vectors that are geometrically closest to the query vector. This allows the system to retrieve the most contextually relevant passages from millions of records in milliseconds.

### Handing Context to the Generative Model
The retrieved passages are not shown to the user immediately. Instead, they are bundled together with the original user query into a "prompt" and sent to a generative model like GPT. This prompt essentially says: *"Using only the following provided context, answer this question."* By feeding the generative model these specific "facts" retrieved by BERT, you significantly reduce the risk of hallucinations and ensure the model has access to data it wasn't originally trained on.



### Concrete Application: Technical Support for Specialized Engineering
A perfect application for BERT-powered RAG is a **Technical Documentation Assistant** for a specialized field like microelectronics or power systems. If an engineer asks, *"How do I optimize the duty cycle for a specific PWM controller?"*, a standard GPT might give a generic answer. However, a RAG system can use BERT to search the company’s internal datasheets and previous troubleshooting logs to find the exact electrical specifications and pass that specific data to GPT. This creates a tool that provides highly accurate, company-specific technical support without needing to retrain the entire model.

---

**As you move forward with these models, would you like to see how to implement a basic vector search using a library like FAISS?**